# Week 9: My first neural network (PyTorch)

For this week I'm using the **diabetes prediction dataset** (100k patients, predicting whether someone has diabetes from things like age, BMI, HbA1c and blood glucose). I picked it because I already know the columns and the class imbalance, so I can focus on the neural network part instead of fighting with the data.

**What's in here**
1. What a neural network is, in my own words
2. A tiny PyTorch warm-up (one neuron, one weight update by hand)
3. Loading + prepping the data
4. Building and training a small network
5. How well it did, and what I think about it

## 1. What a neural network is (my own words)

After watching StatQuest and reading through the PyTorch blitz, this is how I'd explain it:

- A **neuron** is really just a small function. It takes some numbers in, multiplies each one by a **weight**, adds them up, adds a **bias**, and then squishes the result through an **activation function** (I use ReLU, which just turns negatives into 0).
- A **layer** is a bunch of neurons side by side. A network is layers stacked so the output of one layer is the input of the next. The layers in the middle are called hidden layers.
- At the start the weights are random, so the network's predictions are basically garbage.
- **Training** is a loop: (1) push the data through and get predictions (forward pass), (2) measure how wrong they are with a **loss** function, (3) work out how much each weight contributed to the error (**backpropagation**, which PyTorch does for me with `.backward()`), (4) nudge every weight a tiny bit in the direction that makes the loss smaller (**gradient descent**). The size of the nudge is the **learning rate**.
- Repeat that thousands of times and the weights slowly end up somewhere useful. So it "learns" by being wrong a lot and correcting itself a little each time.

The part that clicked for me is that it's the same idea as linear/logistic regression (weights times inputs), just stacked, with the activation functions in between letting it learn curvier patterns.

## 2. Warm-up: one weight, one update

Before the real data, I wanted to see "weights update during training" happen on something so small I can check it by hand. Here the model is just `prediction = w * x`, and I want it to predict 6 when x is 3 (so the perfect weight is 2). I start with w = 1.

In [2]:
import torch

w = torch.tensor(1.0, requires_grad=True)   # requires_grad = "please track this so you can differentiate it later"
x = torch.tensor(3.0)
target = torch.tensor(6.0)

pred = w * x                    # forward pass -> 3.0
loss = (pred - target) ** 2     # how wrong are we? (3 - 6)^2 = 9
loss.backward()                 # backprop: computes d(loss)/d(w)

print("prediction:", pred.item())
print("loss:", loss.item())
print("gradient of loss wrt w:", w.grad.item())   # 2 * (3 - 6) * 3 = -18

lr = 0.01
with torch.no_grad():
    w -= lr * w.grad            # the actual "learning" step
print("new w:", w.item())       # 1 - 0.01 * (-18) = 1.18

: 

The gradient is negative (-18), which basically tells me "increasing w would lower the loss", so the update pushes w up from 1.0 to 1.18, closer to the ideal 2. A real network does exactly this, just for thousands of weights at once and over many rounds.

## 3. Data

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

torch.manual_seed(42)
np.random.seed(42)

df = pd.read_csv("diabetes_prediction_dataset.csv")
print(df.shape)
df.head()

(100000, 9)


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [4]:
print(df.isna().sum().sum(), "missing values")
print()
print(df["diabetes"].value_counts())
print()
print(df["diabetes"].mean().round(3), "= share of patients with diabetes")

0 missing values

diabetes
0    91500
1     8500
Name: count, dtype: int64

0.085 = share of patients with diabetes


Only about 8.5% of the patients have diabetes, so the classes are really imbalanced. That matters later: a model that just says "no diabetes" for everyone would already be right ~91.5% of the time, so plain accuracy isn't going to be a very honest score here.

### Preprocessing
- `gender` and `smoking_history` are text, so I one-hot encode them (neural nets only take numbers).
- I scale the numeric columns (age, bmi, HbA1c, glucose). Without this, glucose (80 to 300) would dominate something like HbA1c (3.5 to 9) just because the numbers are bigger, and training gets slower and weirder.
- I split first and fit the scaler on the **training set only**, so nothing from the test set leaks in.

In [1]:
df = pd.get_dummies(df, columns=["gender", "smoking_history"], drop_first=True, dtype=float)

X = df.drop(columns="diabetes")
y = df["diabetes"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y   # stratify keeps the 8.5% ratio in both splits
)

num_cols = ["age", "bmi", "HbA1c_level", "blood_glucose_level"]
scaler = StandardScaler()
X_train = X_train.copy()
X_test = X_test.copy()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print("features:", list(X.columns))
print("train:", X_train.shape, " test:", X_test.shape)

NameError: name 'pd' is not defined

In [ ]:
# pandas -> PyTorch tensors
X_train_t = torch.tensor(X_train.values.astype("float32"))
X_test_t  = torch.tensor(X_test.values.astype("float32"))
y_train_t = torch.tensor(y_train.values.astype("float32")).unsqueeze(1)   # shape (n, 1) to match the model output
y_test_t  = torch.tensor(y_test.values.astype("float32")).unsqueeze(1)

train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)   # the model sees the data in mini-batches of 256

## 4. The model

Keeping it small: 13 inputs, a hidden layer of 16 neurons, another of 8, and 1 output neuron. The output is a raw score; `BCEWithLogitsLoss` applies the sigmoid inside the loss (so I don't put a sigmoid in the model itself, and I apply it manually later when I want probabilities).

I'm using Adam as the optimizer. From what I understood it's gradient descent with some extra tricks that adjust the step size for each weight, which usually trains faster than plain SGD without me tuning much.

In [ ]:
n_features = X_train_t.shape[1]

model = nn.Sequential(
    nn.Linear(n_features, 16),
    nn.ReLU(),
    nn.Linear(16, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(model)
print("number of weights + biases:", sum(p.numel() for p in model.parameters()))

Quick check on that parameter count: (13×16 + 16) + (16×8 + 8) + (8×1 + 1) = 224 + 136 + 9 = 369. So the whole "brain" is 369 numbers that get adjusted during training.

### Training loop

In [ ]:
epochs = 20
train_losses, test_losses = [], []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()          # clear old gradients (they pile up otherwise)
        out = model(xb)                # 1. forward pass
        loss = loss_fn(out, yb)        # 2. how wrong?
        loss.backward()                # 3. backprop: gradient for every weight
        optimizer.step()               # 4. update the weights
        running_loss += loss.item() * len(xb)

    train_loss = running_loss / len(train_ds)

    model.eval()
    with torch.no_grad():
        test_loss = loss_fn(model(X_test_t), y_test_t).item()

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    print(f"epoch {epoch+1:2d} | train loss {train_loss:.4f} | test loss {test_loss:.4f}")

NameError: name 'model' is not defined

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, epochs + 1), train_losses, label="train")
plt.plot(range(1, epochs + 1), test_losses, label="test")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Loss going down as the weights get updated")
plt.legend()
plt.show()

The loss dropping is basically the "learning" part. If the test loss had started going back up while the train loss kept falling, that would mean overfitting (memorizing the training data), so it's worth keeping an eye on that.

## 5. Results

In [ ]:
model.eval()
with torch.no_grad():
    probs = torch.sigmoid(model(X_test_t)).numpy().ravel()   # raw score -> probability
preds = (probs >= 0.5).astype(int)
y_true = y_test.values

print("neural net accuracy:", round(accuracy_score(y_true, preds), 4))
print("'always say no diabetes' baseline:", round(1 - y_true.mean(), 4))
print()
print("confusion matrix (rows = actual, cols = predicted):")
print(confusion_matrix(y_true, preds))
print()
print(classification_report(y_true, preds, target_names=["no diabetes", "diabetes"]))

## 6. What I take from this

**How it works, short version:** the network starts with random weights, makes predictions on batches of patients, measures the error with the loss, uses backprop to figure out which way each of the 369 weights should move, and updates them a little with the optimizer. After 20 passes over the data it's much better than random.

**What the results say:** the accuracy looks really high (mine was around 97%), but the baseline of always guessing "no diabetes" already gets ~91.5%, so the real story is in the confusion matrix and the recall for the diabetes class. The model is very precise when it says someone has diabetes, but it misses a decent chunk of the people who actually have it (recall was roughly 70% when I ran it). For a medical problem, missing real cases is the worse kind of mistake, so I wouldn't trust this as is.

**Things I'd try next:**
- Give the diabetes class more weight in the loss (`pos_weight` in `BCEWithLogitsLoss`) so mistakes on those patients count more.
- Lower the 0.5 threshold so the model flags more people as possible diabetes cases.
- Compare against logistic regression from earlier weeks. HbA1c and glucose are so predictive here that I wouldn't be surprised if a simple model does about as well.

**What I got out of the week:** neural nets felt like a black box before, but the training loop is only four steps (forward, loss, backward, step). The scary-looking part is mostly just that same loop being run a huge number of times.